# TikTok Recipe Pipeline — Notebook de test

In [1]:
from pathlib import Path
import os, sys
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").exists():
    candidate = Path("/mnt/data/upd/tiktok-data-pipeline")
    if (candidate / "scripts").exists():
        REPO_ROOT = candidate
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "scripts"))
print("Repo root:", REPO_ROOT)

Repo root: c:\Users\fayss\OneDrive - chadstudent.org\Documents\Portfolio_Recipe\tiktok-data-pipeline


In [ ]:
# %pip install -r requirements.txt

In [ ]:
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env", override=True)
3load_dotenv(REPO_ROOT / ".env.example", override=False)
for key in ["SNOWFLAKE_USER","SNOWFLAKE_PASSWORD","SNOWFLAKE_ACCOUNT","SNOWFLAKE_WAREHOUSE","SNOWFLAKE_DB","SNOWFLAKE_SCHEMA_BRONZE","SNOWFLAKE_SCHEMA_SILVER","SNOWFLAKE_SCHEMA_GOLD","SNOWFLAKE_ROLE","OPENROUTER_API_KEY","OPENROUTER_MODEL"]:
    value = os.getenv(key)
    masked = None if value is None else (value[:4] + "***" if len(value) > 4 else "***")
    print(f"{key}: {masked}")

SNOWFLAKE_USER: open***
SNOWFLAKE_PASSWORD: mZjX***
SNOWFLAKE_ACCOUNT: IPLV***
SNOWFLAKE_WAREHOUSE: PORT***
SNOWFLAKE_DB: TIKT***
SNOWFLAKE_SCHEMA_BRONZE: BRON***
SNOWFLAKE_SCHEMA_SILVER: SILV***
SNOWFLAKE_SCHEMA_GOLD: ***
SNOWFLAKE_ROLE: agen***
OPENROUTER_API_KEY: sk-o***
OPENROUTER_MODEL: goog***


In [4]:
from scripts.common import get_snowflake_connection
BRONZE_SCHEMA = os.getenv("SNOWFLAKE_SCHEMA_BRONZE", "BRONZE")
SILVER_SCHEMA = os.getenv("SNOWFLAKE_SCHEMA_SILVER", "SILVER")
with get_snowflake_connection(schema=BRONZE_SCHEMA) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT CURRENT_ACCOUNT(), CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_ROLE(), CURRENT_WAREHOUSE()")
        print(cur.fetchone())

DatabaseError: 250001 (08001): Failed to connect to DB: IPLVEHQ-KL36670.snowflakecomputing.com:443. Incorrect username or password was specified.

In [ ]:
import pandas as pd
query = f'''
SELECT RAW_ID, TITLE, DESCRIPTION, URL_TIKTOK, SOURCE_FILE, RECORD_HASH, INGESTED_AT
FROM {BRONZE_SCHEMA}.BRONZE_TIKTOK_RECIPES
ORDER BY INGESTED_AT DESC
LIMIT 10
'''
with get_snowflake_connection(schema=BRONZE_SCHEMA) as conn:
    df_bronze = pd.read_sql(query, conn)
df_bronze

In [ ]:
from scripts.enrich_silver import fetch_unprocessed_rows
rows = fetch_unprocessed_rows(limit=5)
print(f"Rows to enrich: {len(rows)}")
rows[:2]

In [ ]:
sample_description = rows[0]["DESCRIPTION"] if rows else "Creamy mushroom pasta with garlic, parmesan and spinach. Easy vegetarian dinner."
sample_description

In [ ]:
import requests
from scripts.enrich_silver import ask_openrouter
with requests.Session() as session:
    enrichment = ask_openrouter(sample_description, session=session)
enrichment

In [ ]:
enrichment["structured"]

In [ ]:
from pprint import pprint
import json
if rows:
    row = rows[0]
    structured = enrichment["structured"]
    payload = {
        "raw_id": row["RAW_ID"],
        "title": row["TITLE"],
        "description": row["DESCRIPTION"],
        "url_tiktok": row["URL_TIKTOK"],
        "recipe_language": structured["lang"],
        "is_vegetarian": structured["is_veg"],
        "cuisine_style": structured["cuisine"],
        "main_ingredient": structured["ingredient"],
        "processing_confidence": enrichment["confidence"],
        "model_name": os.getenv("OPENROUTER_MODEL"),
        "llm_raw_response": json.dumps(enrichment["raw_response"]),
        "record_hash": row["RECORD_HASH"],
    }
    pprint(payload)
else:
    print("Aucune ligne Bronze non enrichie disponible.")

In [ ]:
# Décommente pour écrire UNE ligne en Silver
# from scripts.enrich_silver import upsert_silver_row
# if rows:
#     upsert_silver_row(rows[0], enrichment)
#     print("Upsert Silver OK pour RAW_ID =", rows[0]["RAW_ID"])

In [ ]:
query_silver = f'''
SELECT SILVER_ID, RAW_ID, ORIGINAL_TITLE, URL_TIKTOK, RECIPE_LANGUAGE, IS_VEGETARIAN, CUISINE_STYLE, MAIN_INGREDIENT, PROCESSING_CONFIDENCE, MODEL_NAME, PROCESSED_AT, RECORD_HASH
FROM {SILVER_SCHEMA}.SILVER_TIKTOK_RECIPES
ORDER BY PROCESSED_AT DESC
LIMIT 10
'''
with get_snowflake_connection(schema=SILVER_SCHEMA) as conn:
    df_silver = pd.read_sql(query_silver, conn)
df_silver

Ne décommente l'upsert réel qu'une fois le test connexion + OpenRouter validé.